<a href="https://colab.research.google.com/github/lseidy/BI-Data-Sience/blob/main/APS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# APS 2


 O Whisper sofre com limitações de ruídos e sotaques, gerando transcrições as vezes sem o entedimento do contexto da fala. Essa a maior falha é evidenciada pelo termo "endereço de sempre". O Whisper não opera com memória, apenas transcreve o som com precisão, mas repassa o problema para as próximas etapas. O uso de IA generativa acaba então por gerar a situação de não possuir instruções rigorosas ou injeção de contexto, tornando o modelo passível de  alucinação, aceitando a frase literal ou endereços incompletos sem disparar alertas operacionais. Isso demonstra que modelos de linguagem não são validadores logísticos. Para rodar em produção, a arquitetura exige uma forte camada de pré e pós-processamento no backend, onde o ID do cliente deve ser cruzado com um banco de dados para resgatar histórico. Além disso, regras de negócio validam a integridade do pedido antes de enviá-lo à operação.

# Setup


In [32]:
# Instala o openai-whisper.
# A biblioteca traz o modelo, o pipeline de áudio e tudo mais que precisamos.
!pip install -q openai-whisper

In [33]:
import whisper
import time
import os
from IPython.display import Audio, display

In [43]:
TAMANHO = "small"
modelo = whisper.load_model(TAMANHO)

print(f"✅ Modelo Whisper-{TAMANHO} carregado.")

✅ Modelo Whisper-small carregado.


#Audios

In [35]:
# Clona os áudios direto do repo público (--depth 1 = só último commit, rápido).

import os

if not os.path.exists("audios"):
    !git clone --depth 1 https://github.com/atbender/pln-au04-audios.git audios
else:
    print("✓ Pasta 'audios' já existe — pulando clone.")

print("\n📁 Áudios disponíveis:")
for f in sorted(os.listdir("audios")):
    if f.endswith(".mp3"):
        print(f"  - {f}")


✓ Pasta 'audios' já existe — pulando clone.

📁 Áudios disponíveis:
  - aps_01_simples.mp3
  - aps_02_multi.mp3
  - aps_03_restaurante.mp3
  - aps_04_endereco_impreciso.mp3
  - aps_05_ambiguo.mp3
  - aula5_duvida.mp3
  - pratica_01_pizza.mp3
  - pratica_02_acai.mp3
  - pratica_03_cancela.mp3
  - pratica_04_reclamacao.mp3
  - pratica_05_gaucho.mp3


In [36]:
!pip install -q google-generativeai

from google.colab import userdata
import google.generativeai as genai
import json

genai.configure(api_key=userdata.get("GOOGLE_API_KEY"))

# Gemini 2.5 Flash com saída forçada em JSON (sem isso, às vezes ele mete ```json em volta).
modelo_llm = genai.GenerativeModel(
    "gemini-2.5-flash",
    generation_config={"response_mime_type": "application/json"}
)



In [37]:
PROMPT_EXTRAÇÃO = """Você é o sistema de processamento de pedidos por voz do aplicativo RápidoJá.
Extraia as informações do pedido a partir da transcrição do áudio do cliente.

Regras Críticas:
1. Se uma informação não for dita explicitamente no áudio, preencha o valor como null (sem aspas).
2. Devolva estritamente o JSON no formato abaixo, sem nenhum texto adicional.

Formato Esperado:
{
  "pedido": {
    "items": [
      {
        "nome": "string (ex: pizza, refrigerante)",
        "tamanho": "string ou null",
        "sabor": "string ou null",
        "quantidade": inteiro (se não dito, assuma 1),
        "observacoes": "string ou null"
      }
    ],
    "endereco": "string ou null",
    "referencia": "string ou null",
    "restaurante_mencionado": "string ou null",
    "forma_pagamento": "string ou null"
  }
}

Transcrição do cliente: "<<TRANSCRICAO>>"
"""

In [49]:
ARQUIVO = "audios/aps_05_ambiguo.mp3"      # pedido simples
#ARQUIVO = "audios/pratica_02_acai.mp3"       # multi-itens
#ARQUIVO = "audios/pratica_03_cancela.mp3"    # cancelamento
# ARQUIVO = "audios/pratica_04_reclamacao.mp3" # reclamação de atraso
# ARQUIVO = "audios/pratica_05_gaucho.mp3"     # regionalismo gaúcho

# Ouve antes
display(Audio(ARQUIVO))

# Depois transcreve
texto = transcrever(ARQUIVO, modelo)

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📁 Arquivo: audios/aps_05_ambiguo.mp3
🌐 Idioma detectado: pt
⏱  Tempo de transcrição: 13.8s

📝 Transcrição:
   Bota uma pizza grande, metade calabresa metade portuguesa, borda receada de xé da, endereço é o de sempre. Rua Lobo da Costa 200


## Processamento e geração das transcricções

In [64]:
arquivos_audio = ["audios/aps_01_simples.mp3", "audios/aps_02_multi.mp3", "audios/aps_03_restaurante.mp3", "audios/aps_04_endereco_impreciso.mp3", "audios/aps_05_ambiguo.mp3"]
resultados_finais = []

print("🎧 Iniciando processamento dos áudios...\n")

for i, arquivo in enumerate(arquivos_audio):
    id_audio = i + 1

    try:
        # Passo A: Whisper escuta o áudio
        print(f"[{id_audio}/5] Transcrevendo {arquivo}...")
        transcricao = modelo.transcribe(arquivo)
        texto_falado = transcricao["text"].strip()

        # Passo B: Gemini extrai os dados
        print(f"[{id_audio}/5] Extraindo entidades com Gemini...")
        prompt_pronto = PROMPT_EXTRAÇÃO.replace("<<TRANSCRICAO>>", texto_falado)
        resposta_llm = modelo_llm.generate_content(prompt_pronto)

        # Passo C: Montagem do JSON final
        dados_pedido = json.loads(resposta_llm.text)

        json_final = {
            "audio_id": id_audio,
            "transcricao_whisper": texto_falado,
            "pedido": dados_pedido["pedido"]
        }

        resultados_finais.append(json_final)

        # Salva o arquivo individual
        nome_json = f"pedido_{id_audio}.json"
        with open(nome_json, "w", encoding="utf-8") as f:
            json.dump(json_final, f, ensure_ascii=False, indent=2)

        print(f"✅ {nome_json} gerado com sucesso!\n")

    except FileNotFoundError:
        print(f"❌ Erro: O arquivo {arquivo} não foi encontrado no Colab. Faça o upload.\n")
    except Exception as e:
        print(f"❌ Erro ao processar o áudio {id_audio}: {e}\n")



🎧 Iniciando processamento dos áudios...

[1/5] Transcrevendo audios/aps_01_simples.mp3...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


[1/5] Extraindo entidades com Gemini...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


❌ Erro ao processar o áudio 1: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 7.082742036s.

[2/5] Transcrevendo audios/aps_02_multi.mp3...
[2/5] Extraindo entidades com Gemini...


...

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


❌ Erro ao processar o áudio 3: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 38.238373323s.

[4/5] Transcrevendo audios/aps_04_endereco_impreciso.mp3...
[4/5] Extraindo entidades com Gemini...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


❌ Erro ao processar o áudio 4: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 25.073011004s.

[5/5] Transcrevendo audios/aps_05_ambiguo.mp3...
[5/5] Extraindo entidades com Gemini...


❌ Erro ao processar o áudio 5: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 11.196483296s.



In [63]:
# Mostra o resultado do Easter Egg (Áudio 5) para confirmar
if len(resultados_finais) == 5:
    print("-" * 50)
    print("👀 RESULTADO DO ÁUDIO 5 (EASTER EGG):")
    print(json.dumps(resultados_finais[1], indent=2, ensure_ascii=False))

--------------------------------------------------
👀 RESULTADO DO ÁUDIO 5 (EASTER EGG):
{
  "audio_id": 2,
  "transcricao_whisper": "Manda dois burger com batata grande, sem cebola num deles e um suco de laranja natural. É pra Avenida Bento Gonçalves 1456. Apartamento 1203.",
  "pedido": {
    "items": [
      {
        "nome": "burger",
        "tamanho": null,
        "sabor": null,
        "quantidade": 2,
        "observacoes": "sem cebola em um dos burgers"
      },
      {
        "nome": "batata",
        "tamanho": "grande",
        "sabor": null,
        "quantidade": 2,
        "observacoes": null
      },
      {
        "nome": "suco",
        "tamanho": null,
        "sabor": "laranja natural",
        "quantidade": 1,
        "observacoes": null
      }
    ],
    "endereco": "Avenida Bento Gonçalves 1456",
    "referencia": "Apartamento 1203",
    "restaurante_mencionado": null,
    "forma_pagamento": null
  }
}
